# Excel Extraction — Assigning a Specific Agent

Demonstrates two ways to pin a job to a specific agent (or agent pool) when submitting an `@istari:extract` job against an Excel `.xlsx` file using the `open_spreadsheet` tool.

| Option | Approach | When to use |
|--------|----------|-------------|
| **1** | Drop to `platform.client.add_job()` directly | One-off, no library changes needed |
| **2** | Extend `JobDefinition` with agent fields | Repeated use, keeps the fluent style |

### Prerequisites

- An **Istari Digital Platform account** and a **Personal Access Token**.
- An agent with the `open_spreadsheet` module and access to `@istari:extract`.
- The sample file `Group3-UAS-Requirements.xlsx` in the same directory as this notebook.

### Credentials

Create a `.env` file next to this notebook:

```
ISTARI_REGISTRY_URL=https://...paste your platform's registry URL here...
ISTARI_PERSONAL_ACCESS_TOKEN=...paste your token here...
```

## 1 · Connect

In [1]:
from pathlib import Path

from istari_fluent import IstariPlatform, JobDefinition, JobView

platform = IstariPlatform.from_env()

report = platform.client.readiness_check()
assert report.healthy, f"Platform reports unhealthy: {report}"

print(platform)

/Users/craighahn/Documents/GitHub/cookbook/istari-digital-client-cookbook/fluent/.venv/lib/python3.11/site-packages/istari_digital_client/log_utils.py:32: UserWarning: SDK is incompatible with Istari Registry v10.15.2 (affected APIs: v2_main:Systems). Please update your SDK to match the server version.
  result = func(*args, **kwargs)
2026-05-12 18:02:23 -  istari_digital_client.compatibility:process_response_headers:81 - WARNING - SDK is incompatible with Istari Registry v10.15.2 (affected APIs: v2_main:Systems). Please update your SDK to match the server version.


IstariPlatform connected to https://fileservice-v2.demo.istari.app


## 2 · Configure

In [6]:
XLSX_PATH       = Path.cwd() / "Group3-UAS-Requirements.xlsx"
DISPLAY_NAME    = "Group3-UAS-Requirements.xlsx"
EXTERNAL_ID     = "agent-assignment-uas-requirements-demo"

# Set this to the agent ID you want to target (see Section 3 below).
# Leave as None to skip the agent-assigned jobs and just browse available agents.
TARGET_AGENT_ID = "7a0653be-6f09-447d-857c-457df658abcb"   # e.g. "agt-abc123"

assert XLSX_PATH.exists(), f"File not found: {XLSX_PATH}"
print(f"Input file:  {XLSX_PATH}")
print(f"Agent ID:    {TARGET_AGENT_ID or '(not set — browse agents in Section 3 first)'}")

Input file:  /Users/craighahn/Documents/GitHub/cookbook/istari-digital-client-cookbook/samples/Group3-UAS-Requirements.xlsx
Agent ID:    7a0653be-6f09-447d-857c-457df658abcb


## 3 · Browse available agents

List the agent pools the connected user belongs to, then list the agents in each pool.
This reflects what the platform will actually allow you to target — only agents in your
pools are selectable. Set `TARGET_AGENT_ID` in the config cell above, then re-run from
Section 4 onwards.

In [3]:
pools = platform.client.list_agent_pools()

print(f"Agent pools accessible to this user: {pools.total}\n")

for pool in pools.items:
    print(f"Pool: {pool.name!r}  id={pool.id}")

    members = platform.client.list_agent_pool_agents(agent_pool_id=pool.id, size=100)
    if not members.items:
        print("  (no agents)\n")
        continue

    print(f"  {'AGENT ID':<40} {'DISPLAY NAME':<30} {'OS':<20} {'STATUS':<25} MODULES")
    print(f"  {'-'*120}")
    for m in members.items:
        a = m.agent
        display_name  = a.display_name.display_name if a.display_name else ""
        status        = a.status.name if a.status else "Unknown"
        host_os       = a.host_os or ""
        module_names  = (
            [mv.module_name for mv in a.modules.module_versions]
            if a.modules else []
        )
        has_module    = "open_spreadsheet" in module_names
        module_tag    = "  [open_spreadsheet ✓]" if has_module else f"  {module_names}"
        print(f"  {a.id:<40} {display_name:<30} {host_os:<20} {str(status):<25}{module_tag}")
    print()

# Agents with status="Idle" are ready to accept a new job.
# Copy an agent ID from above and paste it into TARGET_AGENT_ID in the config cell.

Agent pools accessible to this user: 1

Pool: 'test1'  id=930b0425-b6d9-496c-85ed-b149a32f12ad
  AGENT ID                                 DISPLAY NAME                   OS                   STATUS                    MODULES
  ------------------------------------------------------------------------------------------------------------------------
  7a0653be-6f09-447d-857c-457df658abcb     loving-will-2462               RHEL 8               AgentStatusName.IDLE       ['@istari:open_pdf', '@istari:open_spreadsheet', '@ntop:geometry']



## 4 · Upload the spreadsheet as a Model resource

In [4]:
model = platform.upload_model(
    XLSX_PATH,
    external_id=EXTERNAL_ID,
    display_name=DISPLAY_NAME,
)
print(f"Uploaded model {model.id}")
print(model)

Uploaded model ec060441-6a1d-4c90-81c5-18e439123a37
Model('Group3-UAS-Requirements.xlsx', id=ec060441-6a1d-4c90-81c5-18e439123a37, file=773a8ec1-d6b6-4d31-b0a4-28707664f93f, rev=646366a5-5209-44a0-a61a-ed57da1bb1f9)


## 5 · Option 1 — assign via `platform.client.add_job()`

Bypass `JobDefinition` entirely and call `add_job()` on the raw client.
This exposes every parameter the API supports, including `assigned_agent_id`
and `assigned_agent_pool_id`.

After submission, wrap the returned `Job` in a `JobView` to get `.wait()`,
`.get_products()`, and the rest of the fluent interface.

In [ ]:
assert TARGET_AGENT_ID, "Set TARGET_AGENT_ID in the config cell before running this section."

job1_raw = platform.client.add_job(
    model_id=model.id,
    function="@istari:extract",
    tool_name="open_spreadsheet",
    assigned_agent_id=TARGET_AGENT_ID,
    # assigned_agent_pool_id="<pool-id>",  # alternative: assign to a pool
)

job1 = JobView(_job=job1_raw, _client=platform.client)
print(f"Submitted job {job1.id} assigned to agent {TARGET_AGENT_ID}; polling...")

job1.wait(
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"\nJob 1 finished: {job1.status}")

products1 = job1.get_products()
print(f"Products: {[p.name for p in products1]}")

Submitted job 0511192b-43b7-42b8-bd22-d05623f9b2c1 assigned to agent 7a0653be-6f09-447d-857c-457df658abcb; polling...
  [Pending] id=0511192b-43b7-42b8-bd22-d05623f9b2c1
  [Pending] id=0511192b-43b7-42b8-bd22-d05623f9b2c1
  [Pending] id=0511192b-43b7-42b8-bd22-d05623f9b2c1
  [Pending] id=0511192b-43b7-42b8-bd22-d05623f9b2c1
  [Pending] id=0511192b-43b7-42b8-bd22-d05623f9b2c1
  [Pending] id=0511192b-43b7-42b8-bd22-d05623f9b2c1
  [Pending] id=0511192b-43b7-42b8-bd22-d05623f9b2c1
  [Pending] id=0511192b-43b7-42b8-bd22-d05623f9b2c1
  [Pending] id=0511192b-43b7-42b8-bd22-d05623f9b2c1


## 6 · Option 2 — extend `JobDefinition`

For repeated use, subclass `JobDefinition` to add the agent fields, then
provide a small helper that calls `add_job()` and returns a `JobView`.
This keeps the submission call site as clean as the standard fluent style
without patching the library.

If you need this across multiple notebooks, move the class and helper into
a shared module (e.g. `istari_fluent/istari_utils.py`) and wire
`assigned_agent_id` through `_submit_job_impl`.

In [9]:
from pydantic import Field as PydanticField


class AgentJobDefinition(JobDefinition):
    """JobDefinition extended with agent/pool targeting."""
    assigned_agent_id: str | None = PydanticField(default=None)
    assigned_agent_pool_id: str | None = PydanticField(default=None)


def submit_job(platform: IstariPlatform, model_id: str, defn: AgentJobDefinition) -> JobView:
    """Submit an AgentJobDefinition and return a JobView ready to .wait()."""
    job_raw = platform.client.add_job(
        model_id=model_id,
        function=defn.function,
        tool_name=defn.tool_name,
        tool_version=defn.tool_version,
        operating_system=defn.operating_system,
        parameters=defn.build_parameters(),
        assigned_agent_id=defn.assigned_agent_id,
        assigned_agent_pool_id=defn.assigned_agent_pool_id,
    )
    return JobView(_job=job_raw, _client=platform.client)

In [10]:
assert TARGET_AGENT_ID, "Set TARGET_AGENT_ID in the config cell before running this section."

extract_on_agent = AgentJobDefinition(
    function="@istari:extract",
    tool_name="open_spreadsheet",
    assigned_agent_id=TARGET_AGENT_ID,
    # assigned_agent_pool_id="<pool-id>",
)

job2 = submit_job(platform, model.id, extract_on_agent)
print(f"Submitted job {job2.id} assigned to agent {TARGET_AGENT_ID}; polling...")

job2.wait(
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"\nJob 2 finished: {job2.status}")

products2 = job2.get_products()
print(f"Products: {[p.name for p in products2]}")

Submitted job a6bafbf4-4f4b-4b80-b33f-f953b7d2929a assigned to agent 7a0653be-6f09-447d-857c-457df658abcb; polling...
  [Pending] id=a6bafbf4-4f4b-4b80-b33f-f953b7d2929a
  [Pending] id=a6bafbf4-4f4b-4b80-b33f-f953b7d2929a
  [Pending] id=a6bafbf4-4f4b-4b80-b33f-f953b7d2929a
  [Pending] id=a6bafbf4-4f4b-4b80-b33f-f953b7d2929a
  [Pending] id=a6bafbf4-4f4b-4b80-b33f-f953b7d2929a
  [Pending] id=a6bafbf4-4f4b-4b80-b33f-f953b7d2929a
  [Pending] id=a6bafbf4-4f4b-4b80-b33f-f953b7d2929a
  [Pending] id=a6bafbf4-4f4b-4b80-b33f-f953b7d2929a
  [Pending] id=a6bafbf4-4f4b-4b80-b33f-f953b7d2929a
  [Pending] id=a6bafbf4-4f4b-4b80-b33f-f953b7d2929a
  [Pending] id=a6bafbf4-4f4b-4b80-b33f-f953b7d2929a
  [Pending] id=a6bafbf4-4f4b-4b80-b33f-f953b7d2929a
  [Canceled] id=a6bafbf4-4f4b-4b80-b33f-f953b7d2929a


KeyboardInterrupt: 

## Verify in the UI

1. **Jobs / Activity** — Both jobs should show `open_spreadsheet / @istari:extract`.
2. **Job detail** — Each job should show the assigned agent under its execution details.

## Optional · Archive the model

In [ ]:
model.archive()